# C3-A7 · Hands-on: a conditional diffusion model for PV disaggregation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcjene/ai4gd-c3-diffusion/blob/main/AI4GD_C3-A7.ipynb)

**AI4GreenDeal · Course C3, Conditional diffusion modeling for energy disaggregation · about 120 min**

In C3-A5 you saw how a conditional diffusion model (cDDPM) generates the PV behind a smart meter, and in C3-A6 how
to turn its samples into prediction intervals. Now you write the key pieces yourself, train the model and evaluate it.

| § | Section | min |
|---|---|---|
| 0 | Setup | 5 |
| 1 | Data and conditions | 10 |
| 2 | Forward process | 12 |
| 3a | Inside the U-Net | 15 |
| 3 | Training | 15 |
| 4 | Sampling | 18 |
| 5 | From samples to estimates | 20 |
| 6 | What does the model listen to? | 20 |
| 7 | Wrap-up | 5 |

**How the exercises work.** Replace every `____` with your code and run the cell. `ex1.check(...)` tells you whether
it is right, and if not, what is probably wrong. Stuck? Uncomment `ex1.hint()` or `ex1.solution()`. If a check
fails, the notebook still goes on: the next cells use the reference solution and say so. Questions are answered with
`q1.check("B")`. Each section starts with a **catch-up cell**: after a runtime restart, run it and carry on.

> The numbers in this notebook are **illustrations, not results of the paper**: the course models use a different
> noise schedule (β up to 0.2 instead of 0.02), and we evaluate on a few days only.

**Runtime.** Choose a GPU: *Runtime → Change runtime type → T4 GPU*. Without a GPU the notebook still runs: training
is replaced by the course checkpoint, each day gets 20 samples instead of 50, §6 uses 3 days and the optional Ex8c
is skipped.

**Notation** (as in C3-A5 and C3-A6): $x_0$ one day of PV, $T = 48$ half-hours; $x_k$ the noisy profile at
diffusion step $k = 1 \dots K$, $K = 100$; $c$ the conditions; $\varepsilon_\theta(x_k, k, c)$ the noise predicted by
the network; $\bar\alpha_k$ the signal level; $pv_i$, $\hat{pv}_i$ true and estimated PV at half-hour $i$.

In [ ]:
# §0 Setup: install the course package (on Colab), check the GPU, download the data (14 MB)
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/marcjene/ai4gd-c3-diffusion"], check=True)

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import ai4gd_c3 as c3
from ai4gd_c3 import data, model, plots, reference
from ai4gd_c3.exercises import *   # ____, ex1 ... ex9, q1 ... q8, progress, catch_up

GPU = c3.DEVICE != "cpu"
print("device:", torch.cuda.get_device_name() if c3.DEVICE == "cuda" else c3.DEVICE)
if not GPU:
    print("No GPU: §3 loads the course checkpoint instead of training, 20 samples per day instead of 50,\n"
          "§6 uses 3 days, Ex8c is skipped.")
D = data.load()
print(f"data: {len(D['customers'])} Ausgrid customers ({D['is_train'].sum()} for training), {len(D['pv'])} days")

## 1 · Data and conditions (≈ 10 min)

The model generates one day of PV, $x_0$, guided by three conditions $c$: the irradiance $\mathrm{GHI}_d$ and the
net consumption of the same day, $\mathrm{NC}_d$, and of the previous day, $\mathrm{NC}_{d-1}$. The data are the
Ausgrid Solar Home dataset (Sydney, July 2012 to June 2013, 30 min) with GHI from Open-Meteo.

As in C3-A5, PV is divided by the installed capacity $C$ and net consumption by the household's maximum export:

$$x_0 = \frac{PV\ [\mathrm{kW}]}{C\ [\mathrm{kWp}]}, \qquad
\mathrm{NC}^{\mathrm{pu}} = \frac{\mathrm{NC}\ [\mathrm{kW}]}{\text{max. export}\ [\mathrm{kW}]}, \qquad
\mathrm{GHI}\ \text{in kW/m}^2.$$

We follow customer 134 from C3-A5/A6.

In [ ]:
# Catch-up §1
S = catch_up(1)
d = S.day   # customer 134, clear day 2013-01-04: values in kW (pv_kw, nc_kw, ...) and in pu
print(f"customer {d['customer']}, {d['date']}: C = {d['capacity_kwp']:.2f} kWp, max. export {d['max_export_kw']:.2f} kW")

C3-A6 rounded this household to 1.5 kWp; here we use the measured value, so kW values differ by about 1 %.

**Ex1.** Convert the day from kW to the model's per-unit values.

In [ ]:
def normalise(pv_kw, nc_kw, capacity_kwp, max_export_kw):
    # kW -> pu: PV / installed capacity, NC / maximum export
    pv_pu = ____
    nc_pu = ____
    return pv_pu, nc_pu

ex1.check(normalise)
# ex1.hint()
# ex1.solution()

In [ ]:
normalise = ex1.use(normalise)
pv_pu, nc_pu = normalise(d["pv_kw"], d["nc_kw"], d["capacity_kwp"], d["max_export_kw"])
print(f"peak PV {d['pv_kw'].max():.2f} kW = {pv_pu.max():.2f} pu;  lowest NC {d['nc_kw'].min():.2f} kW = {nc_pu.min():.2f} pu")
plots.day(d)

**Look at your data.** Around noon NC drops below zero: the household exports, and that is the PV imprint on the
meter. Over the whole dataset, two values look odd at first:

In [ ]:
print(f"largest PV value: {D['pv'].max():.2f} pu of installed capacity")
print(f"largest NC value: {D['nc'].max():.1f} pu of maximum export")

PV above 1 pu happens when a system produces more than its listed capacity (nameplate understating the array,
metadata errors or metering effects). And NC of 25 pu?

**q1.** Why can NC reach 25 pu?
**A** a measurement error · **B** the household rarely exports, so its maximum export is small · **C** NC counts PV twice

In [ ]:
q1.check(____)   # A, B or C

## 2 · Forward process (≈ 12 min)

To learn to remove noise, the model needs noisy examples. We mix the real PV with Gaussian noise. The course uses a
linear schedule from $\beta_1 = 10^{-4}$ to $\beta_K = 0.2$ with $K = 100$ ([1] uses 0.02, for which
$\bar\alpha_K \approx 0.36$ and $x_K$ is not pure noise):

$$\alpha_k = 1 - \beta_k, \qquad \bar\alpha_k = \prod_{j=1}^{k} \alpha_j, \qquad
x_k = \sqrt{\bar\alpha_k}\, x_0 + \sqrt{1 - \bar\alpha_k}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I).$$

**Index convention.** The equations count $k$ from 1, Python counts from 0. We store the schedule with $K + 1$
entries and put $\beta_0 = 0$, $\alpha_0 = \bar\alpha_0 = 1$ at index 0. Then `abar[k]` is exactly $\bar\alpha_k$ of
C3-A5 for $k = 1 \dots K$, and step 0 is $x_0$ itself. (Inside, the network still receives $k - 1$, because the course
checkpoints were trained that way; you never have to think about it.)

In [ ]:
# Catch-up §2
S = catch_up(1)
d = S.day
x0 = d["pv_pu"]   # [48], pu

**Ex2.** Build the schedule.

In [ ]:
def linear_schedule(K, beta_1=1e-4, beta_K=0.2):
    # tensors of length K + 1; index 0 holds beta_0 = 0 and alpha_0 = abar_0 = 1
    beta = ____     # 0, then beta_1 ... beta_K evenly spaced
    alpha = ____
    abar = ____     # cumulative product of alpha
    return beta, alpha, abar

ex2.check(linear_schedule)
# ex2.hint()
# ex2.solution()

In [ ]:
linear_schedule = ex2.use(linear_schedule)
beta, alpha, abar = linear_schedule(model.K)
print(f"abar_0 = {abar[0]:.3f}, abar_1 = {abar[1]:.5f}, abar_K = {abar[model.K]:.1e}")

**Ex3.** Jump straight to step $k$ with the closed form. `k` holds one step per profile.

In [ ]:
def q_sample(x0, k, eps, abar):
    # x0, eps: [B, T]; k: [B] integers in 1..K
    a = ____        # abar_k of each profile, shaped [B, 1] so that it broadcasts over T
    return ____     # sqrt(abar_k) x_0 + sqrt(1 - abar_k) eps

ex3.check(q_sample)
# ex3.hint()
# ex3.solution()

In [ ]:
q_sample = ex3.use(q_sample)
torch.manual_seed(0)
eps = torch.randn(1, 48)
plots.forward(x0, abar, [0, 10, 27, 50, 100], lambda k: q_sample(x0[None], torch.tensor([k]), eps, abar)[0])

**q2.** Look at the right-hand plot: at about which step does the noise share overtake the signal share?
**A** about 10 · **B** about 27 · **C** 90 or later

In [ ]:
q2.check(____)   # A, B or C

In [ ]:
k_cross = int((abar < 0.5).nonzero()[0])   # first k with abar_k < 0.5, where sqrt(1 - abar_k) > sqrt(abar_k)
print("the shares cross at k =", k_cross)

## 3a · Inside the U-Net (≈ 15 min)

The noise predictor $\varepsilon_\theta(x_k, k, c)$ is the 1D U-Net of Fig. 3 in [1] (C3-A5, screen 6). The conditions
are stacked with $x_k$ as input channels; the step $k$ enters every modulated convolution through FiLM.

| Block | What it does |
|---|---|
| `input` | stack $x_k$ and the 3 conditions as channels |
| `init_conv` | Conv1d, kernel 3, to $B_{ch} = 128$ channels |
| `encoder_1` | modulated convolution, doubles the channels (copy → skip 1) |
| `encoder_2` | max pooling ÷2, modulated convolution, doubles the channels (copy → skip 2) |
| `bottleneck` | max pooling ÷2, two modulated convolutions (to 1024 channels and back) |
| `decoder_2` | transposed convolution ×2, join skip 2, modulated convolution to 256 channels |
| `decoder_1` | transposed convolution ×2, join skip 1, modulated convolution to 128 channels |
| `output` | Conv1d to 1 channel, squeezed: $\varepsilon_\theta$ has the shape of $x_k$ |

FiLM in each modulated convolution, with $(\gamma, \delta)$ computed from the step embedding of $k$:
$h \leftarrow \mathrm{ReLU}\big(h \cdot (1 + \gamma) + \delta\big)$.

**Ex4a.** Predict the output shape of every block for a batch of $B = 16$ profiles.

In [ ]:
shapes = {
    "input":      (16, 4, 48),   # given: x_k and 3 conditions as channels, T = 48
    "init_conv":  ____,
    "encoder_1":  ____,
    "encoder_2":  ____,
    "bottleneck": ____,
    "decoder_2":  ____,
    "decoder_1":  ____,
    "output":     ____,
}
ex4a.check(shapes)
# ex4a.hint()

**Ex4b.** Complete the U-Net: **blank 1** is FiLM, **blank 2** joins the decoder input with its skip connection.
Keep all layer names: the check loads the course weights (`pv_all.pt`) into your class and compares the outputs.

In [ ]:
from ai4gd_c3.model import TimestepEmbedding   # sinusoidal embedding of k (size 32) -> MLP -> 128


class ModulatedConv1D(nn.Module):
    # Conv1d (kernel 3) whose output is scaled and shifted according to the step k (FiLM)
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1)
        self.film = nn.Linear(time_emb_dim, out_ch * 2)   # one (scale, shift) pair per output channel

    def forward(self, x, t_emb):
        h = self.conv(x)                                   # [B, out_ch, L]
        # Blank 1 (FiLM): scale and shift from self.film(t_emb), each [B, out_ch];
        # then h * (1 + scale) + shift, broadcast over the length L
        scale, shift = ____
        h = ____

        return F.relu(h)


class UNet1D(nn.Module):
    def __init__(self, in_ch, base_channels, time_embed_dim, diffusion_steps, depth=2):
        super().__init__()
        self.time_embedding = TimestepEmbedding(time_embed_dim, base_channels)
        self.init_conv = nn.Conv1d(in_ch, base_channels, kernel_size=3, padding=1)
        self.enc_blocks = nn.ModuleList()
        self.pool = nn.MaxPool1d(kernel_size=2)
        ch, skip_ch = base_channels, []
        for _ in range(depth):
            self.enc_blocks.append(ModulatedConv1D(ch, ch * 2, base_channels))
            skip_ch.append(ch)
            ch *= 2
        self.bot1 = ModulatedConv1D(ch, ch * 2, base_channels)
        self.bot2 = ModulatedConv1D(ch * 2, ch, base_channels)
        self.up_blocks, self.dec_blocks = nn.ModuleList(), nn.ModuleList()
        for s in reversed(skip_ch):
            self.up_blocks.append(nn.ConvTranspose1d(ch, s, kernel_size=2, stride=2))
            self.dec_blocks.append(ModulatedConv1D(ch + s, s, base_channels))
            ch = s
        self.final_conv = nn.Conv1d(ch, 1, kernel_size=3, padding=1)

    def forward(self, x, k, c):
        # x: x_k [B, T]; k: steps 1..K [B]; c: conditions [B, 3, T]
        t_emb = self.time_embedding(k - 1)                          # the checkpoints use 0-based steps
        x = self.init_conv(torch.cat([x.unsqueeze(1), c], dim=1))   # input -> initial convolution
        skips = []
        for enc in self.enc_blocks:                                 # encoder levels 1, 2
            x = F.relu(enc(x, t_emb))
            skips.append(x)
            x = self.pool(x)
        x = F.relu(self.bot1(x, t_emb))                             # bottleneck
        x = F.relu(self.bot2(x, t_emb))
        for up, dec in zip(self.up_blocks, self.dec_blocks):        # decoder levels 2, 1
            skip = skips.pop()
            x = up(x)
            x = ____                                                # blank 2: join x and its skip

            x = F.relu(dec(x, t_emb))
        return self.final_conv(x).squeeze(1)                        # output: eps_theta, [B, T]


ex4b.check(UNet1D)
# ex4b.hint()
# ex4b.solution()

In [ ]:
UNet = ex4b.use(UNet1D)
net = model.new_model("all", cls=UNet)
print(f"{sum(p.numel() for p in net.parameters()):,} parameters")   # 5,421,569, as in [1]

**q3.** Each convolution sees 3 neighbouring values, and each pooling halves the length. How many input half-hours
does one bottleneck value see (after both bottleneck convolutions)?
**A** 5 · **B** 10 · **C** 28

In [ ]:
q3.check(____)   # A, B or C

## 3 · Training (≈ 15 min)

One training iteration (Algorithm 1 in [1], C3-A5 screen 4):

1. take a batch of real days $x_0$ with their conditions $c$;
2. draw one step $k \sim \mathcal{U}\{1, \dots, K\}$ and noise $\varepsilon \sim \mathcal{N}(0, I)$ per profile;
3. noise the profiles: $x_k = \sqrt{\bar\alpha_k}\, x_0 + \sqrt{1 - \bar\alpha_k}\, \varepsilon$;
4. loss $= \text{mean of } \big(\varepsilon - \varepsilon_\theta(x_k, k, c)\big)^2$; one Adam step.

The course model repeats this 2000 times, with batch 256 and learning rate $10^{-3}$, over the 107 training households.

In [ ]:
# Catch-up §3
S = catch_up(2)
beta, alpha, abar = S.schedule
q_sample = ex3.use(globals().get("q_sample", reference.q_sample))
UNet = ex4b.use(globals().get("UNet1D", model.UNet1D))

**Ex4c.** Write the loss of one training iteration (steps 2–4).

In [ ]:
def diffusion_loss(model, x0, c, abar, K):
    # x0: [B, T], c: [B, 3, T]
    k = ____          # one step per profile, integers 1..K (device=x0.device)
    eps = ____        # Gaussian noise with the shape of x0
    x_k = ____        # your q_sample
    return ____       # mean squared error between eps and model(x_k, k, c)

ex4c.check(diffusion_loss)
# ex4c.hint()
# ex4c.solution()

Now train the model with all three conditions. With a T4 GPU this takes a few minutes. `TRAIN_IN_NOTEBOOK = False`
loads the course checkpoint `pv_all.pt` instead: the same configuration, already trained.

In [ ]:
TRAIN_IN_NOTEBOOK = True

diffusion_loss = ex4c.use(diffusion_loss)
TRAINED = TRAIN_IN_NOTEBOOK and GPU
if TRAINED:
    net = model.new_model("all", cls=UNet)
    t0 = time.time()
    losses = model.train(net, diffusion_loss, abar)   # 2000 steps, batch 256, Adam lr 1e-3, seed 0
    print(f"trained in {(time.time() - t0) / 60:.1f} min")
    plots.losses(losses)
else:
    print("↪ loading the course checkpoint pv_all.pt instead of training" + ("" if GPU else " (no GPU)"))
    net = model.load_pretrained("all", cls=UNet)

The loss falls fast in the first few hundred steps and then flattens. It stays noisy because every batch mixes
different days and different steps $k$: predicting the noise is easy at large $k$ and hard at small $k$.

## 4 · Sampling (≈ 18 min)

Generation starts from pure noise $x_K \sim \mathcal{N}(0, I)$ and applies the network once per step, from $k = K$
down to 1 (Algorithm 2 in [1], C3-A5 screen 5):

$$x_{k-1} = \frac{1}{\sqrt{\alpha_k}} \left( x_k - \frac{\beta_k}{\sqrt{1 - \bar\alpha_k}}\,
\varepsilon_\theta(x_k, k, c) \right) + \sigma_k z, \qquad \sigma_k = \sqrt{\beta_k},\quad z \sim \mathcal{N}(0, I),$$

with no fresh noise at the last step ($z = 0$ for $k = 1$).

In [ ]:
# Catch-up §4
S = catch_up(4)
beta, alpha, abar = schedule = S.schedule
net = globals().get("net", S.model)   # your trained model (the course checkpoint after a restart)
TRAINED = globals().get("TRAINED", False) and net is not S.model
course = S.model                      # course checkpoint pv_all.pt
d = S.day

**Ex5.** One reverse step. `k` is a Python integer here; `z` is given so that the check is reproducible.

In [ ]:
def reverse_step(x_k, eps_hat, k, beta, alpha, abar, z):
    mean = ____
    return ____       # add sigma_k z, except at the last step k = 1

ex5.check(reverse_step)
# ex5.hint()
# ex5.solution()

Generate 50 samples (20 without a GPU) for customer 134 on the clear day, with your model and with the course model (same initial
noise). They should look alike, but not identical: training is random. (If you did not train in §3, your model *is*
the course model, and you see one panel.)

In [ ]:
reverse_step = ex5.use(reverse_step)
mine = model.sample(net, d["c"], schedule, reverse_step)   # [samples, 48], pu, raw model output
panels = [(mine, d, "your model" if TRAINED else "course model pv_all.pt")]
if TRAINED:   # same initial noise, course checkpoint
    panels.append((model.sample(course, d["c"], schedule, reverse_step), d, "course model pv_all.pt"))
plots.compare(1, len(panels), [(s, dd, f"{title}, raw samples") for s, dd, title in panels])

**Ex5b. Watch the reverse process.** At every step the network's noise estimate also gives a guess of the final
profile:

$$\hat{x}_0 = \frac{x_k - \sqrt{1 - \bar\alpha_k}\; \varepsilon_\theta(x_k, k, c)}{\sqrt{\bar\alpha_k}}.$$

**q4. Predict first:** on the clear day, at about which $k$ does the daily PV shape first become visible in
$\hat{x}_0$? **A** about 80 · **B** about 40 · **C** about 10

In [ ]:
q4.check(____)   # A, B or C

In [ ]:
def predict_x0(x_k, eps_hat, k, abar):
    return ____

ex5b.check(predict_x0)

In [ ]:
predict_x0 = ex5b.use(predict_x0)
ks = [100, 80, 60, 40, 20, 10]
days = {label: data.day(data.CUSTOMER, date) for label, date in data.DAYS_134.items()}
final, kept = model.sample(course, torch.stack([dd["c"] for dd in days.values()]), schedule, reverse_step,
                           record=ks)                                # course model, days a, b, c


def med_corr(profiles, truth):  # median over the samples of the correlation with the true PV
    return torch.stack([torch.corrcoef(torch.stack([p, truth]))[0, 1] for p in profiles]).nanmedian().item()


print(f"median correlation with the true PV over {final.shape[1]} samples")
print("            " + "".join(f"k = {k:<5}" for k in ks + [0]))
for i, (label, dd) in enumerate(days.items()):
    hat = [med_corr(predict_x0(kept[k][0][i], kept[k][1][i], k, abar), dd["pv_pu"]) for k in ks]
    raw = [med_corr(kept[k][0][i], dd["pv_pu"]) for k in ks]
    last = med_corr(final[i], dd["pv_pu"])
    print(f"day {label}  x̂_0  " + "".join(f"{v:<9.2f}" for v in hat + [last]))
    print(f"       x_k  " + "".join(f"{v:<9.2f}" for v in raw + [last]))

b = list(days).index("b")
plots.trajectory(ks + [0], [[kept[k][0][b, 0] for k in ks] + [final[b, 0]],
                            [predict_x0(kept[k][0][b, 0], kept[k][1][b, 0], k, abar) for k in ks] + [final[b, 0]]],
                 days["b"]["pv_pu"])

$\hat{x}_0$ shows the shape long before $x_k$ does: the network guesses the overall profile early, and the later steps
refine the detail. On the overcast day (c) the shape appears much later: there is little PV to recover.

**Ex6. Post-processing.** The raw samples can be negative and are not zero at night. As in C3-A5: clamp at 0, and set
PV to 0 wherever GHI = 0.

In [ ]:
def postprocess(samples, ghi):
    # samples: [N, T] pu; ghi: [T]. Return a new tensor (do not modify samples in place).
    return ____

ex6.check(postprocess)
# ex6.hint()

In [ ]:
postprocess = ex6.use(postprocess)
plots.compare(1, len(panels), [(postprocess(s, d["ghi"]), dd, title) for s, dd, title in panels])

## 5 · From samples to estimates (≈ 20 min)

From here on we use the course model `pv_all.pt`, so that the numbers match C3-A6, and work in kW. Definitions as in
C3-A6 and [1], Section 3.2. $N = 48$ for RMSE; $N_{sun}$ counts the daylight half-hours ($\mathrm{GHI} > 0$), and
$C$ is the installed capacity.

$$\mathrm{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (pv_i - \hat{pv}_i)^2}, \qquad
\mathrm{PICP} = \frac{100}{N_{sun}} \sum_{i=1}^{N_{sun}} c_i, \qquad
\mathrm{PINAW} = \frac{1}{N_{sun}} \sum_{i=1}^{N_{sun}} \frac{\hat{pv}_i^{\,q_u} - \hat{pv}_i^{\,q_\ell}}{C},$$

with $c_i = 1$ if $pv_i$ lies inside the interval, and the pinball loss of quantile $q$, averaged over
$q = 0.1, \dots, 0.9$ and the daylight half-hours:

$$PL_i^q = \begin{cases} (1 - q)\,(\hat{pv}_i^{\,q} - pv_i) & \text{if } pv_i < \hat{pv}_i^{\,q} \\
q\,(pv_i - \hat{pv}_i^{\,q}) & \text{otherwise.} \end{cases}$$

In [ ]:
# Catch-up §5: post-processed samples (50, or 20 without GPU) of the course model for days a (drop), b (clear), c (overcast)
S = catch_up(5)
days = S.days
samples_kw = {label: S.samples[label] * dd["capacity_kwp"] for label, dd in days.items()}   # [samples, 48] in kW
d, s = days["b"], samples_kw["b"]

**Ex7.** Point estimate (median), central prediction interval and RMSE.

In [ ]:
def point_and_band(samples, level=80):
    # samples: [N, T] -> median, lower, upper, each [T], for the central `level` % interval
    q = ____                          # the three quantile levels, as fractions, in a tensor
    median, lower, upper = ____       # torch.quantile over the samples
    return median, lower, upper


def rmse(pv, pv_hat):
    return ____


ex7.check(point_and_band, rmse)
# ex7.hint()
# ex7.solution()

In [ ]:
point_and_band, rmse = ex7.use(point_and_band, rmse)
med, lo, up = point_and_band(s, 80)
plots.band(d, med, lo, up, 80)
for label, dd in days.items():
    print(f"day {label}: RMSE of the median {rmse(dd['pv_kw'], point_and_band(samples_kw[label])[0]):.3f} kW (all 48 half-hours)")

**Ex8.** Calibration (PICP) and sharpness (PINAW), on daylight half-hours only.

In [ ]:
def picp_pinaw(pv, lower, upper, ghi, capacity):
    # pv, lower, upper in kW, [T]; ghi [T]; capacity in kWp. Returns PICP in % and PINAW in kW/kWp.
    sun = ____
    inside = ____
    picp = ____
    pinaw = ____
    return picp, pinaw

ex8.check(picp_pinaw)
# ex8.hint()

In [ ]:
picp_pinaw = ex8.use(picp_pinaw)
for label, dd in days.items():
    for level in (50, 80):
        _, lo, up = point_and_band(samples_kw[label], level)
        p, w = picp_pinaw(dd["pv_kw"], lo, up, dd["ghi"], dd["capacity_kwp"])
        print(f"day {label}, {level} % PI:  PICP {p:5.1f} %   PINAW {w:.3f} kW/kWp")

One day is not enough to judge calibration: by chance a single day can land above or below the promised level
(C3-A6). Over 300 days, the 80 % PI of this model covered 86.5 % of the true values.

**Ex8c (optional). How many samples are enough?** We draw $N$ = 10, 50 and 200 samples, five times each, and look at
how much the 10 %, 50 % and 90 % quantiles at noon and the PICP change between repeats.

**q6. Predict first:** which stabilises most slowly as $N$ grows?
**A** the median · **B** the 10 % and 90 % quantiles · **C** all equally

In [ ]:
q6.check(____)   # A, B or C

In [ ]:
if not GPU:
    print("Ex8c skipped without a GPU.")
else:
    S8 = catch_up(4)
    noon = data.TIME.index("12:00")
    rows = {10: [], 50: [], 200: []}
    for r in range(5):
        s200 = postprocess(model.sample(S8.model, d["c"], S8.schedule, reverse_step, n=200, seed=100 + r), d["ghi"])
        s200 = s200 * d["capacity_kwp"]
        for n in rows:   # the first n of 200 independent samples
            med, lo, up = point_and_band(s200[:n], 80)
            p, _ = picp_pinaw(d["pv_kw"], lo, up, d["ghi"], d["capacity_kwp"])
            rows[n].append((lo[noon].item(), med[noon].item(), up[noon].item(), p.item()))
    plots.repeats(rows)

**Ex9.** The pinball loss scores all quantiles at once.

In [ ]:
def pinball(pv, q_pred, q, ghi):
    # pv [T]; q_pred [Q, T] predicted quantiles; q [Q] levels; mean over q and daylight half-hours
    err = ____
    loss = ____       # (1 - q) * (q_pred - pv) where pv < q_pred, q * (pv - q_pred) otherwise
    return ____

ex9.check(pinball)
# ex9.hint()

In [ ]:
pinball = ex9.use(pinball)
q = torch.arange(1, 10) / 10
for label, dd in days.items():
    pl = pinball(dd["pv_kw"], torch.quantile(samples_kw[label], q, dim=0), q, dd["ghi"])
    print(f"day {label}: pinball loss {pl:.4f} kW = {pl / dd['capacity_kwp']:.4f} kW/kWp")

## 6 · What does the model listen to? (≈ 20 min)

**Ex8b.** We give the model with all three conditions inputs that contradict each other (all real values for
customer 134):

- **(a)** the overcast day (c) with the GHI of the clear day (b), keeping its real $\mathrm{NC}_d$ and $\mathrm{NC}_{d-1}$;
- **(a′)** the overcast day with the clear day's $\mathrm{NC}_d$, keeping its real GHI;
- **(b)** the clear day with $\mathrm{NC}_d$ drawn as a straight line from 07:00 to 18:00, so the household no longer
  appears to export (the values stay within the observed range).

**q5. Predict first:** will the samples follow the sky (GHI) or the meter (NC)?
**A** the sky · **B** the meter · **C** in between

In [ ]:
q5.check(____)   # A, B or C

In [ ]:
# Catch-up §6
S = catch_up(6)
schedule, models = S.schedule, S.models
reverse_step = ex5.use(globals().get("reverse_step", reference.reverse_step))
postprocess = ex6.use(globals().get("postprocess", reference.postprocess))
point_and_band, rmse = ex7.use(globals().get("point_and_band", reference.point_and_band),
                               globals().get("rmse", reference.rmse))
picp_pinaw = ex8.use(globals().get("picp_pinaw", reference.picp_pinaw))
pinball = ex9.use(globals().get("pinball", reference.pinball))

In [ ]:
clear, dull = (data.day(data.CUSTOMER, data.DAYS_134[label]) for label in "bc")
nc_flat = clear["nc_pu"].clone()
nc_flat[14:37] = torch.linspace(nc_flat[14].item(), nc_flat[36].item(), 23)   # 07:00 ... 18:00
cases = {   # name: (conditions GHI_d, NC_d, NC_d-1;  day whose true PV we show)
    "overcast, real":                (dull["c"], dull),
    "(a) overcast + clear-sky GHI":  (torch.stack([clear["ghi"], dull["nc_pu"], dull["nc_lag1_pu"]]), dull),
    "(a′) overcast + clear NC_d":    (torch.stack([dull["ghi"], clear["nc_pu"], dull["nc_lag1_pu"]]), dull),
    "clear, real":                   (clear["c"], clear),
    "(b) clear, NC_d without export": (torch.stack([clear["ghi"], nc_flat, clear["nc_lag1_pu"]]), clear),
}
raw = model.sample(models["all"], torch.stack([c for c, _ in cases.values()]), schedule, reverse_step)
panels = []
for i, (name, (c, dd)) in enumerate(cases.items()):
    s = postprocess(raw[i], c[0])
    med, lo, up = point_and_band(s * dd["capacity_kwp"], 80)
    print(f"{name:32s} median energy {med.sum() / 2:5.2f} kWh (true {dd['pv_kw'].sum() / 2:.2f})"
          f"   mean width of the 80 % PI {(up - lo)[c[0] > 0].mean():.3f} kW")
    panels.append((s, dd, name))
plots.compare(2, 3, panels)

The samples follow the meter. With the clear-sky GHI, the overcast day's PV stays low and the spread grows (a);
with the clear day's export, the overcast day suddenly gets a lot of PV (a′); and without the midday export, the
clear day loses most of its PV (b).

The model **learned from data, not physics**. In the 2012–13 Ausgrid data a clear day almost always comes with a
midday export, so the model learned "no export ⇒ no PV". On 21 test households the pattern held for most; it was
weak for the few that export little even on clear days (customers 12, 25, 149 and 276), where GHI weighs more.

**q7.** In (b) the model is confidently wrong. Which household would this hurt in practice?
**A** one with a battery or a large daytime load · **B** one with east-facing panels · **C** any household on an overcast day

In [ ]:
q7.check(____)   # A, B or C

**Compare the three course models** (GHI only, NC only, all three) on several test days, with your functions from §5.
MASE is computed by the package (C3-A6: below 1 means better than repeating the previous half-hour).

In [ ]:
days6 = [data.day(cust, date) for cust, date in data.experiment_days()[: 8 if GPU else 3]]
c6 = torch.stack([dd["c"] for dd in days6])
q = torch.arange(1, 10) / 10
rows, samples6 = [], {}
for name, m in models.items():
    raw = model.sample(m, c6, schedule, reverse_step)
    for i, dd in enumerate(days6):
        s = postprocess(raw[i], dd["ghi"]) * dd["capacity_kwp"]
        samples6[name, i] = s
        med, lo, up = point_and_band(s, 80)
        p, w = picp_pinaw(dd["pv_kw"], lo, up, dd["ghi"], dd["capacity_kwp"])
        pl = pinball(dd["pv_kw"], torch.quantile(s, q, dim=0), q, dd["ghi"]) / dd["capacity_kwp"]
        rows.append({"model": name, "day": f"{dd['customer']} {dd['date']}", "RMSE (kW)": float(rmse(dd["pv_kw"], med)),
                     "MASE": float(reference.mase(dd["pv_kw"], med)), "PICP80 (%)": float(p),
                     "PINAW80 (kW/kWp)": float(w), "pinball (kW/kWp)": float(pl)})

import pandas as pd
table = pd.DataFrame(rows)
summary = table.groupby("model", sort=False).agg({"RMSE (kW)": "median", "MASE": "median", "PICP80 (%)": "mean",
                                                  "PINAW80 (kW/kWp)": "mean", "pinball (kW/kWp)": "mean"})
print(f"{len(days6)} test days; RMSE and MASE: median over days, the rest: mean")
display(summary.round(3))
plots.compare(3, 3, [(samples6[n, i] / days6[i]["capacity_kwp"], days6[i], f"{n}, {days6[i]['date']}")
                     for n in models for i in range(3)])

**q8.** Which model scores best on the pinball loss, which rewards calibration and sharpness together?
**A** all three conditions · **B** GHI only · **C** NC only

In [ ]:
q8.check(____)   # A, B or C

## 7 · Wrap-up (≈ 5 min)

In [ ]:
progress()

**Key idea.** A conditional diffusion model learns to predict the noise added to real profiles. Run backwards from
pure noise, and guided at every step by smart-meter and weather data, it generates many plausible PV profiles; their
quantiles give point estimates and prediction intervals, judged by calibration and sharpness. What the model knows
comes from its training data: conditions are evidence, weighed as the data taught it, not physical rules.

**Optional extensions** (GPU, a few minutes each): change the number of steps $K$ or the width $B_{ch}$, retrain
and compare with the course model. Fewer steps make sampling faster; a narrower network has a quarter of the weights
at half the width (C3-A5, screen 9).

In [ ]:
RUN_EXTENSION = False   # set to True to run
K_NEW, WIDTH = 50, 64

if RUN_EXTENSION and GPU:
    S = catch_up(6)
    linear_schedule = ex2.use(globals().get("linear_schedule", reference.linear_schedule))
    diffusion_loss = ex4c.use(globals().get("diffusion_loss", reference.diffusion_loss))
    UNet = ex4b.use(globals().get("UNet1D", model.UNet1D))
    sched_new = linear_schedule(K_NEW)
    net_new = UNet(4, base_channels=WIDTH, time_embed_dim=32, diffusion_steps=K_NEW, depth=2).to(c3.DEVICE)
    plots.losses(model.train(net_new, diffusion_loss, sched_new[2]))
    d = S.day
    plots.compare(1, 2, [(postprocess(model.sample(net_new, d["c"], sched_new, reverse_step), d["ghi"]), d,
                          f"K = {K_NEW}, B_ch = {WIDTH}"),
                         (postprocess(model.sample(S.models["all"], d["c"], S.schedule, reverse_step), d["ghi"]), d,
                          "course model (K = 100, B_ch = 128)")])

### Sources

[1] M. Jené-Vinuesa, H. Kazmi, M. Aragüés-Peñalba and A. Sumper, "Conditional diffusion modeling for probabilistic
behind-the-meter PV disaggregation," *Energy and AI*, art. no. 100692, 2026, doi: 10.1016/j.egyai.2026.100692.
[2] J. Ho, A. Jain and P. Abbeel, "Denoising diffusion probabilistic models," in *Proc. NeurIPS*, vol. 33,
pp. 6840–6851, 2020.
[3] E. L. Ratnam, S. R. Weller, C. M. Kellett and A. T. Murray, "Residential load and rooftop PV generation: an
Australian distribution network dataset," *Int. J. Sustain. Energy*, vol. 36, no. 8, pp. 787–806, 2017.
[4] P. Zippenfenig, "Open-Meteo.com Weather API," Zenodo, 2023, doi: 10.5281/zenodo.7970649.

**Data:** Ausgrid Solar Home Electricity dataset (Creative Commons Attribution) and Open-Meteo (CC BY 4.0); the
preprocessed subset and the course checkpoints are published under CC BY 4.0 (see `DATA_README.md` in the
repository). Code: MIT licence.